# E-AMP — fingerprint amplitude sweep
**Turns "λ < λ_U" into a detection threshold in physical units.**

Injects device B's fingerprint into device A's training images at known multiples of natural
amplitude, trains one adapter per amplitude, and measures whether K_B re-emerges in generations.

Design follows the review's §7:
- **multiplicative** injection matching the PRNU model: `Y_α = clip[Y ⊙ (1 + α·K̂_B)]`
- amplitudes **α ∈ {1, 1.6, 3, 6, 12}** (α=0 is the existing A_raw arms)
- **spectrally-matched random field Q** at the top amplitude — separates "learns *this* fingerprint"
  from "learns *any* strong fixed-coordinate high-frequency texture"
- **circularly-shifted K_B** at the top amplitude — a real transfer must follow the injected alignment
- injection uses `K̂_B` from split **E1**; measurement uses `K̂_B` from split **E2** (disjoint), so the
  removal/injection error cannot manufacture the effect

**Reuses** the base run's manifest and fingerprints read-only. Writes to its own `EXT_ROOT`
because the protocol hash necessarily changes.

Run order: A0 → A1 → A2 → A3 → A4 → A5. Roughly 5 h unattended on an A100.

## Run all and leave

1. **Add your HF token to Colab Secrets** (key icon, left sidebar) as `HF_TOKEN`, with
   *Notebook access* enabled. Without it the token cell prompts and the run blocks.
2. **Runtime → Change runtime type → A100** (bf16 is required; the preflight halts on fp16).
3. **Runtime → Run all.** Approve the Google Drive mount when it appears — that is the only
   interaction. After that it runs end to end.

Every stage is resumable and skip-if-complete, so if the session drops, just Run all again;
finished work is detected and skipped.

`STAGES` is already set to the full sequence. To run a single stage instead, edit that tuple.


In [ ]:
# %% TOKEN — RUN THIS FIRST. Non-interactive if HF_TOKEN is in Colab Secrets.
#   Colab: key icon in the left sidebar -> add secret named HF_TOKEN -> enable "Notebook access".
#   Without a secret this falls back to a prompt, which BLOCKS an unattended run.
import os, sys, subprocess
try:
    import huggingface_hub  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=False)

_tok = ""
try:
    from google.colab import userdata
    _tok = (userdata.get("HF_TOKEN") or "").strip()
    if _tok: print("[TOKEN] loaded from Colab Secrets")
except Exception:
    pass
if not _tok:
    _tok = os.environ.get("HF_TOKEN", "").strip()
if not _tok:
    import getpass
    print("[TOKEN] no Colab Secret found — prompting (this blocks an unattended run)")
    _tok = getpass.getpass("HF token (SD-3.5 license must be accepted on your account): ").strip()
if not _tok:
    raise SystemExit("No HF token. SD-3.5 is a gated repo and every training/generation "
                     "stage will fail without it.")
os.environ["HF_TOKEN"] = _tok
from huggingface_hub import login
login(token=_tok)
print("[TOKEN] Hugging Face login OK — safe to Run all and walk away")


In [ ]:
# %% A0 — config, env, and base-run reuse
import os, sys, json, glob, re, math, time, gc, shutil, hashlib, subprocess, random
from dataclasses import dataclass, asdict
from concurrent.futures import ThreadPoolExecutor

from google.colab import drive; drive.mount("/content/drive", force_remount=False)
DRIVE = "/content/drive/MyDrive"
if not os.path.isdir(DRIVE):
    raise SystemExit("Drive not mounted — re-run and complete the auth prompt.")

BASE_ROOT = os.path.join(DRIVE, "inv_channel/E_INV_P0_v3")   # the completed C0 run
EXT_ROOT  = os.path.join(DRIVE, "inv_channel/E_AMP")         # this experiment

@dataclass
class CFG:
    STAGES: tuple = ("A0", "A1", "A2", "A3", "A4", "A5")
    HF_MODEL: str = "stabilityai/stable-diffusion-3.5-medium"
    CROP: int = 1024
    N_T: int = 50
    # amplitudes: 0 is the existing A_raw arms in the base run
    ALPHAS: tuple = (1.0, 1.6, 3.0, 6.0, 12.0)
    CTRL_ALPHA: float = 12.0          # amplitude for the Q and shifted controls
    SHIFT: tuple = (137, 251)         # not multiples of 8 — avoids the VAE grid
    LORA_RANK: int = 16
    LORA_TARGETS: tuple = ("to_q","to_k","to_v","to_out.0")
    LR: float = 1e-4; STEPS: int = 2000; BATCH: int = 2; GRAD_ACC: int = 2
    CAPTION: str = "a photograph, sks style"
    SEED: int = 0
    G_PER_ADAPTER: int = 500
    GEN_SEED_BASE: int = 770000
    GEN_STEPS: int = 28; CFG_SCALE: float = 4.5; GEN_BATCH: int = 4
    N_PERM: int = 10000
    GRAD_CKPT: bool = False

C = CFG()
MEAS = C.CROP
CFG_SHA = hashlib.sha256(json.dumps(
    {k: v for k, v in asdict(C).items() if k != "STAGES"}, sort_keys=True, default=str
).encode()).hexdigest()

for d in ["", "train_png", "adapters", "gens", "csv", "meta", "figs", "fingerprints"]:
    os.makedirs(os.path.join(EXT_ROOT, d), exist_ok=True)
CACHE = "/content/e_amp_cache"; os.makedirs(CACHE, exist_ok=True)

def stage(name):
    if name in C.STAGES: return True
    print(f"[{name}] SKIPPED — not in C.STAGES={C.STAGES}"); return False

# ---- arms -------------------------------------------------------------------------
def all_arms():
    a = [(f"amp_{str(x).replace('.','p')}", "amp", x) for x in C.ALPHAS]
    a.append(("ctrl_Q",     "Q",     C.CTRL_ALPHA))     # spectrally-matched random field
    a.append(("ctrl_shift", "shift", C.CTRL_ALPHA))     # circularly-shifted K_B
    return a
def all_tags(): return [t for t, _, _ in all_arms()]
def arm_of(tag): return next(a for a in all_arms() if a[0] == tag)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "PyWavelets", "diffusers>=0.31", "transformers", "peft", "accelerate",
                "safetensors", "sentencepiece", "protobuf"], check=False)
import numpy as np, pywt, torch
import torch.nn.functional as tF
from PIL import Image
import scipy.ndimage as ndi
Image.MAX_IMAGE_PIXELS = None
DEV = "cuda" if torch.cuda.is_available() else "cpu"
TRAIN_DTYPE = torch.bfloat16 if DEV == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
np.seterr(divide="raise", invalid="raise")
_MEASURE_DEV = DEV
if not os.environ.get("HF_TOKEN"):
    raise SystemExit("HF_TOKEN not set — run the TOKEN cell at the top first.")

def HALT(m): raise SystemExit(f"\n{'='*70}\n[HALT] {m}\n{'='*70}")
def append_row(p, hdr, row):
    new = not os.path.exists(p)
    with open(p, "a") as f:
        if new: f.write(",".join(hdr) + "\n")
        f.write(",".join(str(x) for x in row) + "\n")
def done_keys(p, cols):
    if not os.path.exists(p): return set()
    import csv as _c
    with open(p) as f: return {tuple(r[k] for k in cols) for r in _c.DictReader(f)}
def atomic_json_dump(o, p, **kw):
    t = p + ".tmp"; json.dump(o, open(t, "w"), **kw); os.replace(t, p)

# ---- verify the base run and load its artifacts -----------------------------------
import csv as _csv
man = os.path.join(BASE_ROOT, "manifest", "manifest.csv")
if not os.path.exists(man): HALT(f"base manifest not found: {man}")
M = {}
for r in _csv.DictReader(open(man)):
    M.setdefault((r["role"], r["split"]), []).append(r["path"])
def K_base(role, which="E2", comp="K"):
    p = os.path.join(BASE_ROOT, "fingerprints", f"{comp}_{role}_{which}.npy")
    if not os.path.exists(p): HALT(f"base fingerprint missing: {p}")
    return np.load(p)
print(f"[A0] base run OK — A:T={len(M[('A','T')])} A:H={len(M[('A','H')])}")
print(f"[A0] arms: {all_tags()}")
# --- peft/torchao compatibility -----------------------------------------------------
# peft's is_torchao_available() RAISES ImportError when torchao is older than its
# minimum (Colab ships 0.10.0, peft wants >=0.16.0) instead of returning False, which
# kills add_adapter(). Nothing here is quantised, so disable the dispatcher outright.
try:
    import peft.import_utils as _piu
    _piu.is_torchao_available = lambda: False
    try:
        import peft.tuners.lora.torchao as _plt
        _plt.is_torchao_available = lambda: False
    except Exception:
        pass
    print("[ENV] peft torchao dispatcher disabled")
except Exception as _e:
    print("[ENV] torchao patch skipped:", type(_e).__name__)

print(f"[A0] protocol SHA = {CFG_SHA[:16]}  device={DEV} dtype={TRAIN_DTYPE}")

In [ ]:
# %% forensic core — identical to the base run (fp32 measurement path)
def _cp(fp): return os.path.join(CACHE, hashlib.sha1(f"{fp}|{MEAS}".encode()).hexdigest()[:16] + ".npy")

def load_lum_crop(fp, use_cache=True):
    if use_cache and os.path.exists(_cp(fp)): return np.load(_cp(fp))
    with Image.open(fp) as im:
        im = im.convert("RGB"); W, H = im.size
        if min(W, H) < MEAS: HALT(f"{fp}: {W}x{H} < MEAS {MEAS}")
        a = np.asarray(im.crop(((W-MEAS)//2, (H-MEAS)//2, (W-MEAS)//2+MEAS, (H-MEAS)//2+MEAS)), np.float32)
    lum = (0.299*a[...,0] + 0.587*a[...,1] + 0.114*a[...,2]).astype(np.float32)
    if use_cache: np.save(_cp(fp), lum)
    return lum

def load_rgb_crop(fp):
    with Image.open(fp) as im:
        im = im.convert("RGB"); W, H = im.size
        return np.asarray(im.crop(((W-MEAS)//2, (H-MEAS)//2, (W-MEAS)//2+MEAS, (H-MEAS)//2+MEAS)), np.float32)

def _conv2(x, k):
    xt = torch.from_numpy(np.ascontiguousarray(x))[None, None]
    kt = torch.from_numpy(k)[None, None]
    if _MEASURE_DEV == "cuda":
        xt, kt = xt.cuda(), kt.cuda()
        return tF.conv2d(xt, kt, padding=k.shape[0]//2)[0, 0].cpu().numpy()
    return tF.conv2d(xt, kt, padding=k.shape[0]//2)[0, 0].numpy()

def wavelet_residual(img):
    s0 = np.float32(2.0)**2
    co = pywt.wavedec2(img, "db8", level=4, mode="periodization")
    out = [co[0]*0.0]
    for (cH, cV, cD) in co[1:]:
        band = []
        for ch in (cH, cV, cD):
            ch = ch.astype(np.float32); vmin = None
            for w in (3, 5, 7, 9):
                v = np.maximum(_conv2(ch*ch, np.ones((w, w), np.float32)/(w*w)) - s0, 0.0)
                vmin = v if vmin is None else np.minimum(vmin, v)
            band.append(ch*(s0/(vmin + s0)))
        out.append(tuple(band))
    n = pywt.waverec2(out, "db8", mode="periodization")[:img.shape[0], :img.shape[1]]
    r = n - n.mean(); sd = float(r.std())
    if not np.isfinite(sd) or sd == 0.0: HALT("degenerate residual")
    return (r/sd).astype(np.float32)

def ncc0(a, b):
    a = a - a.mean(); b = b - b.mean()
    na = float(np.linalg.norm(a)); nb = float(np.linalg.norm(b))
    return 0.0 if na == 0 or nb == 0 else float((a*b).sum()/(na*nb))

def _ncc_batch(a, B):
    a = a - a.mean(); na = a.norm()
    Bc = B - B.mean(dim=(1, 2), keepdim=True)
    nb = Bc.flatten(1).norm(dim=1)
    return (Bc*a).flatten(1).sum(dim=1) / (na*nb).clamp_min(1e-30)

# ---- injection operators (review §7) ----------------------------------------------
def inject(rgb, K, alpha):
    """Multiplicative PRNU-model injection: Y_alpha = clip[Y * (1 + alpha*K)]."""
    return np.clip(rgb*(1.0 + alpha*K[..., None]), 0, 255).astype(np.float32)

def spectral_match(K, seed=0):
    """Random field with the SAME power spectrum (hence autocorrelation) and variance as K."""
    rng = np.random.default_rng(seed)
    mag = np.abs(np.fft.fft2(K.astype(np.float32)))
    ph = np.fft.fft2(rng.standard_normal(K.shape).astype(np.float32))
    ph = ph/np.maximum(np.abs(ph), 1e-12)
    Q = np.real(np.fft.ifft2(mag*ph)).astype(np.float32)
    Q = Q - Q.mean()
    return (Q*(K.std()/max(Q.std(), 1e-12))).astype(np.float32)

def shift_field(K, dr, dc): return np.roll(np.roll(K, dr, axis=0), dc, axis=1).astype(np.float32)

In [ ]:
# %% A1 — build the injected training sets + the measurement fingerprint bank
#     Also runs the REAL-IMAGE dose-response first: cheap, and it proves the measurement
#     chain responds monotonically to alpha before a single GPU-hour is spent on training.
def train_dir(tag): return os.path.join(EXT_ROOT, "train_png", tag)

def a1_materialise():
    KB_e1 = K_base("B", "E1")            # used ONLY to modify images
    KB_e2 = K_base("B", "E2")            # used ONLY to measure
    Q     = spectral_match(KB_e1, seed=C.SEED)
    KBs   = shift_field(KB_e1, *C.SHIFT)
    print(f"[A1] NCC(K_B_E1, K_B_E2)      = {ncc0(KB_e1, KB_e2):.4f}  (disjoint estimates, same device)")
    print(f"[A1] NCC(Q, K_B_E2)           = {ncc0(Q, KB_e2):+.4f}  (must be ~0)")
    print(f"[A1] NCC(K_B_shift, K_B_E2)   = {ncc0(KBs, KB_e2):+.4f}  (must be ~0)")
    print(f"[A1] sd(Q)/sd(K_B)            = {Q.std()/KB_e1.std():.4f}  (spectral match)")

    # measurement bank: base roles + the two control fields, measured with E2/derived-from-E1
    bank = {r: K_base(r, "E2") for r in ("A", "B", "C", "D", "D2")}
    bank["Bshift"] = shift_field(KB_e2, *C.SHIFT)     # shifted MEASUREMENT fingerprint
    bank["Q"] = Q
    for k, v in bank.items():
        np.save(os.path.join(EXT_ROOT, "fingerprints", f"K_{k}.npy"), v)
    print(f"[A1] measurement bank: {list(bank)}")

    # ---- real-image dose-response (no training) ----
    hdr = ["alpha", "idx", "rho_KB", "rho_KA"]
    p = os.path.join(EXT_ROOT, "csv", "a1_real_dose.csv")
    if not os.path.exists(p):
        for a in (0.0,) + tuple(C.ALPHAS):
            for i, fp in enumerate(M[("A", "H")]):
                rgb = load_rgb_crop(fp)
                img = rgb if a == 0 else inject(rgb, KB_e1, a)
                Y = (0.299*img[...,0] + 0.587*img[...,1] + 0.114*img[...,2]).astype(np.float32)
                W = wavelet_residual(Y)
                append_row(p, hdr, [a, i, f"{ncc0(W, Y*bank['B']):.6e}", f"{ncc0(W, Y*bank['A']):.6e}"])
            print(f"[A1] real dose-response alpha={a} done")
    import pandas as pd
    d = pd.read_csv(p)
    print("\n[A1] REAL-IMAGE DOSE-RESPONSE (the measurement chain's reference curve)")
    for a, g in d.groupby("alpha"):
        print(f"   alpha={a:5.1f}  median rho(->K_B)={g.rho_KB.median():+.5f}   rho(->K_A)={g.rho_KA.median():+.5f}")
    print("   monotone in alpha => the chain is calibrated; generations are compared against this")

    # ---- materialise training sets ----
    files = M[("A", "T")]
    for tag, kind, a in all_arms():
        d_ = train_dir(tag); os.makedirs(d_, exist_ok=True)
        if len(glob.glob(os.path.join(d_, "*.png"))) >= len(files):
            print(f"[A1] skip {tag} (materialised)"); continue
        F = {"amp": KB_e1, "Q": Q, "shift": KBs}[kind]
        for i, fp in enumerate(files):
            out = inject(load_rgb_crop(fp), F, a)
            Image.fromarray(np.clip(out, 0, 255).astype(np.uint8)).save(
                os.path.join(d_, f"{i:04d}.png"), compress_level=1)
        print(f"[A1] materialised {tag}: kind={kind} alpha={a} n={len(files)}")

if stage("A1"): a1_materialise()

In [ ]:
# %% A2 — train one adapter per arm
def adapter_dir(tag): return os.path.join(EXT_ROOT, "adapters", tag)

def _expected(tag):
    _, kind, a = arm_of(tag)
    return {"tag": tag, "kind": kind, "alpha": a, "rank": C.LORA_RANK, "steps": C.STEPS,
            "meas": MEAS, "seed": C.SEED, "config_sha": CFG_SHA, "model": C.HF_MODEL,
            "dtype": str(TRAIN_DTYPE)}

def adapter_is_valid(tag):
    p = os.path.join(adapter_dir(tag), "train_meta.json")
    if not os.path.exists(os.path.join(adapter_dir(tag), "pytorch_lora_weights.safetensors")): return False
    m = json.load(open(p)) if os.path.exists(p) else None
    return m is not None and all(m.get(k) == v for k, v in _expected(tag).items()) and m.get("effective", False)

def _prompt_embeds():
    p = os.path.join(EXT_ROOT, "adapters", "_prompt_embeds.pt")
    if os.path.exists(p):
        d = torch.load(p, map_location=DEV); return d["pe"].to(DEV), d["pp"].to(DEV)
    from diffusers import StableDiffusion3Pipeline
    pipe = StableDiffusion3Pipeline.from_pretrained(C.HF_MODEL, torch_dtype=TRAIN_DTYPE, token=True).to(DEV)
    with torch.no_grad():
        pe, _, pp, _ = pipe.encode_prompt(prompt=C.CAPTION, prompt_2=C.CAPTION, prompt_3=C.CAPTION,
                                          device=DEV, num_images_per_prompt=1,
                                          do_classifier_free_guidance=False)
    torch.save({"pe": pe.cpu(), "pp": pp.cpu()}, p)
    del pipe; gc.collect(); torch.cuda.empty_cache()
    return pe.to(DEV), pp.to(DEV)

def train_arm(tag, pe, pp):
    if adapter_is_valid(tag): print(f"[A2] skip valid {tag}"); return
    if DEV != "cuda": HALT("A2 needs a GPU")
    from diffusers import StableDiffusion3Pipeline
    from peft import LoraConfig
    from peft.utils import get_peft_model_state_dict
    out = adapter_dir(tag)
    if os.path.exists(out): shutil.move(out, out + f"__stale_{int(time.time())}")
    torch.manual_seed(C.SEED); np.random.seed(C.SEED); random.seed(C.SEED)
    try:
        pipe = StableDiffusion3Pipeline.from_pretrained(
            C.HF_MODEL, torch_dtype=TRAIN_DTYPE, token=True,
            text_encoder=None, text_encoder_2=None, text_encoder_3=None,
            tokenizer=None, tokenizer_2=None, tokenizer_3=None).to(DEV)
    except Exception:
        pipe = StableDiffusion3Pipeline.from_pretrained(C.HF_MODEL, torch_dtype=TRAIN_DTYPE, token=True).to(DEV)
        pipe.text_encoder = pipe.text_encoder_2 = pipe.text_encoder_3 = None
    sch, vae, tr = pipe.scheduler, pipe.vae, pipe.transformer
    vae.requires_grad_(False); tr.requires_grad_(False)
    tr.add_adapter(LoraConfig(r=C.LORA_RANK, lora_alpha=C.LORA_RANK,
                              init_lora_weights="gaussian", target_modules=list(C.LORA_TARGETS)))
    if C.GRAD_CKPT: tr.enable_gradient_checkpointing()
    tr.train()
    params = [p for p in tr.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=C.LR, weight_decay=1e-4)
    lrs = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=math.ceil(C.STEPS/C.GRAD_ACC))
    opt.zero_grad(set_to_none=True)
    files = sorted(glob.glob(os.path.join(train_dir(tag), "*.png")))
    if len(files) != C.N_T: HALT(f"{tag}: {len(files)} training PNGs, expected {C.N_T}")
    cache = torch.stack([torch.from_numpy(
        (np.asarray(Image.open(f).convert("RGB"), np.float32).copy()/127.5 - 1.0)).permute(2, 0, 1)
        for f in files])
    sf = float(vae.config.scaling_factor); sh = float(vae.config.shift_factor or 0.0)
    ts, sg = sch.timesteps.to(DEV), sch.sigmas.to(DEV)
    n_ts = int(sch.config.num_train_timesteps)
    if len(sg) != n_ts or len(ts) != n_ts: HALT("scheduler array length mismatch — pin diffusers")
    g = torch.Generator().manual_seed(C.SEED); losses = []; t0 = time.time()
    print(f"\n[A2] {tag}: rank {C.LORA_RANK}, {C.STEPS} steps, {len(files)} imgs @ {MEAS}")
    for st in range(C.STEPS):
        idx = torch.randint(0, cache.shape[0], (C.BATCH,), generator=g)
        px = cache[idx].to(DEV, dtype=TRAIN_DTYPE)
        with torch.no_grad():
            lat = vae.encode(px).latent_dist.sample(); lat = ((lat - sh)*sf).to(TRAIN_DTYPE)
        noise = torch.randn_like(lat)
        u = torch.sigmoid(torch.randn(lat.shape[0], device=DEV, dtype=torch.float32))
        ii = (u*n_ts).long().clamp_(0, n_ts - 1)
        sig = sg[ii].to(lat.dtype).view(-1, 1, 1, 1)
        pred = tr(hidden_states=((1.0 - sig)*lat + sig*noise).to(TRAIN_DTYPE), timestep=ts[ii],
                  encoder_hidden_states=pe.repeat(lat.shape[0], 1, 1).to(TRAIN_DTYPE),
                  pooled_projections=pp.repeat(lat.shape[0], 1).to(TRAIN_DTYPE),
                  return_dict=False)[0]
        loss = tF.mse_loss(pred.float(), (noise - lat).float())/C.GRAD_ACC
        loss.backward()
        if (st + 1) % C.GRAD_ACC == 0:
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); lrs.step(); opt.zero_grad(set_to_none=True)
        losses.append(float(loss.detach())*C.GRAD_ACC)
        if not np.isfinite(losses[-1]): HALT(f"{tag}: non-finite loss at step {st+1}")
        if (st + 1) % 400 == 0:
            el = time.time() - t0
            print(f"    {st+1}/{C.STEPS} loss={np.mean(losses[-400:]):.5f} "
                  f"eta={(el/(st+1))*(C.STEPS-st-1)/60:.1f}min")
    sd = get_peft_model_state_dict(tr)
    bnorm = float(sum(float(v.float().norm()) for k, v in sd.items() if "lora_B" in k))
    if not (bnorm > 1e-6 and all(bool(torch.isfinite(v).all()) for v in sd.values())):
        HALT(f"{tag}: DEAD adapter (||lora_B||={bnorm:.3e})")
    os.makedirs(out, exist_ok=True)
    StableDiffusion3Pipeline.save_lora_weights(save_directory=out, transformer_lora_layers=sd)
    meta = _expected(tag); meta.update({"loss_tail": float(np.mean(losses[-100:])),
                                        "lora_B_norm": bnorm, "effective": True,
                                        "minutes": (time.time()-t0)/60})
    atomic_json_dump(meta, os.path.join(out, "train_meta.json"), indent=2)
    print(f"[A2] saved {tag}: ||lora_B||={bnorm:.2f} tail={meta['loss_tail']:.5f} "
          f"({meta['minutes']:.1f} min)")
    del pipe, tr, vae, cache; gc.collect(); torch.cuda.empty_cache()

if stage("A2"):
    pe, pp = _prompt_embeds()
    for i, t in enumerate(all_tags(), 1):
        print(f"[A2] adapter {i}/{len(all_tags())}: {t}")
        train_arm(t, pe, pp)
    bad = [t for t in all_tags() if not adapter_is_valid(t)]
    if bad: HALT(f"A2 incomplete: {bad}")
    print(f"[A2] PASS — {len(all_tags())} adapters")

In [ ]:
# %% A3 — generation, identical paired seed bank as the base run
def gen_dir(tag): return os.path.join(EXT_ROOT, "gens", tag)
def bank_(): return [(C.CAPTION, C.GEN_SEED_BASE + j) for j in range(C.G_PER_ADAPTER)]

def a3_generate():
    if DEV != "cuda": HALT("A3 needs a GPU")
    from diffusers import StableDiffusion3Pipeline
    b = bank_()
    pipe = StableDiffusion3Pipeline.from_pretrained(C.HF_MODEL, torch_dtype=TRAIN_DTYPE, token=True).to(DEV)
    pipe.set_progress_bar_config(disable=True)
    prev = False
    try:
        for tag in all_tags():
            d_ = gen_dir(tag); os.makedirs(d_, exist_ok=True)
            for _t in glob.glob(os.path.join(d_, "*.tmp")): os.remove(_t)
            miss = [i for i in range(C.G_PER_ADAPTER)
                    if not os.path.exists(os.path.join(d_, f"{i:05d}.png"))]
            if not miss: print(f"[A3] skip complete {tag}"); continue
            if prev: pipe.unload_lora_weights(); prev = False
            pipe.load_lora_weights(adapter_dir(tag)); prev = True
            print(f"[A3] {tag}: {len(miss)} missing")
            for s in range(0, len(miss), C.GEN_BATCH):
                ix = miss[s:s+C.GEN_BATCH]
                gs = [torch.Generator(device=DEV).manual_seed(b[i][1]) for i in ix]
                with torch.inference_mode():
                    imgs = pipe(prompt=[b[i][0] for i in ix], num_inference_steps=C.GEN_STEPS,
                                guidance_scale=C.CFG_SCALE, height=MEAS, width=MEAS,
                                generator=gs).images
                for i, im in zip(ix, imgs):
                    t_ = os.path.join(d_, f"{i:05d}.png.tmp")
                    im.save(t_, format="PNG")   # PIL cannot infer format from .tmp
                    os.replace(t_, os.path.join(d_, f"{i:05d}.png"))
                if (s // C.GEN_BATCH) % 25 == 0:
                    print(f"    {s+len(ix)}/{len(miss)}")
            print(f"[A3] complete {tag}")
    finally:
        del pipe; gc.collect(); torch.cuda.empty_cache()
    print("[A3] PASS")

if stage("A3"): a3_generate()

In [ ]:
# %% A4 — measurement against the full bank (incl. the Q and shifted controls)
BANK_ROLES = ["A", "B", "C", "D", "D2", "Bshift", "Q"]

def a4_measure():
    K = torch.stack([torch.from_numpy(np.load(os.path.join(EXT_ROOT, "fingerprints", f"K_{r}.npy")))
                     for r in BANK_ROLES]).to(_MEASURE_DEV)
    hdr = ["tag", "gen_idx", "K", "rho_mult", "rho_add"]
    p = os.path.join(EXT_ROOT, "csv", "a4_measure.csv")
    done = done_keys(p, ["tag", "gen_idx", "K"])
    pool = ThreadPoolExecutor(max_workers=4)
    try:
        for tag in all_tags():
            paths = [os.path.join(gen_dir(tag), f"{i:05d}.png") for i in range(C.G_PER_ADAPTER)]
            todo = [i for i in range(C.G_PER_ADAPTER)
                    if not all((tag, str(i), r) in done for r in BANK_ROLES)]
            if not todo: print(f"[A4] {tag}: complete"); continue
            fut = {i: pool.submit(load_lum_crop, paths[i], False) for i in todo[:6]}
            t0 = time.time()
            for pos, i in enumerate(todo):
                Z_np = fut.pop(i).result()
                nx = pos + 6
                if nx < len(todo): fut[todo[nx]] = pool.submit(load_lum_crop, paths[todo[nx]], False)
                W = torch.from_numpy(wavelet_residual(Z_np)).to(_MEASURE_DEV)
                Z = torch.from_numpy(Z_np).to(_MEASURE_DEV)
                rm = _ncc_batch(W, Z.unsqueeze(0)*K).cpu().numpy()
                ra = _ncc_batch(W, K).cpu().numpy()
                for j, r in enumerate(BANK_ROLES):
                    if (tag, str(i), r) in done: continue
                    append_row(p, hdr, [tag, i, r, f"{rm[j]:.6e}", f"{ra[j]:.6e}"])
            print(f"[A4] {tag}: {len(todo)} measured ({time.time()-t0:.0f}s)")
    finally:
        pool.shutdown(wait=True); gc.collect()
        if DEV == "cuda": torch.cuda.empty_cache()
    n = len(done_keys(p, ["tag", "gen_idx", "K"]))
    exp = len(all_tags())*C.G_PER_ADAPTER*len(BANK_ROLES)
    if n != exp: HALT(f"A4 incomplete: {n}/{exp}")
    print(f"[A4] PASS — {n}/{exp} rows")

if stage("A4"): a4_measure()

In [ ]:
# %% A5 — dose-response curve, threshold, and the two specificity controls
def a5_analyse():
    import pandas as pd
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    rng = np.random.default_rng(0)
    d = pd.read_csv(os.path.join(EXT_ROOT, "csv", "a4_measure.csv"))
    real = pd.read_csv(os.path.join(EXT_ROOT, "csv", "a1_real_dose.csv"))

    def v(tag, K): return d[(d.tag == tag) & (d.K == K)].sort_values("gen_idx")["rho_mult"].to_numpy(float)
    def sf(x, n=None):
        n = n or C.N_PERM; o = x.mean(); c = 0
        for _ in range(n):
            c += ((x*rng.choice((-1.0, 1.0), len(x))).mean() >= o)
        return (c + 1)/(n + 1), float(o)

    out = {"config_sha": CFG_SHA, "amplitudes": {}, "controls": {}}
    print("── dose-response: does injected K_B re-emerge in generations? ──")
    print(f"{'arm':14s} {'alpha':>6s} {'rho(->K_B)':>12s} {'contrast vs K_A':>16s} {'p':>9s}")
    xs, ys = [], []
    for tag, kind, a in all_arms():
        if kind != "amp": continue
        b = v(tag, "B"); ctr = b - v(tag, "A")     # paired: cancels the generic offset
        p, obs = sf(ctr)
        out["amplitudes"][a] = {"rho_B": float(b.mean()), "contrast": obs, "p": p,
                                "n": int(len(b))}
        xs.append(a); ys.append(obs)
        print(f"{tag:14s} {a:6.1f} {b.mean():12.3e} {obs:16.3e} {p:9.4g}"
              f"{'  <== DETECTED' if p < 0.01 else ''}")

    print("\n── specificity controls at alpha={:.0f} ──".format(C.CTRL_ALPHA))
    for tag, kind, a in all_arms():
        if kind == "amp": continue
        tgt = "Q" if kind == "Q" else "Bshift"
        c_t = v(tag, tgt) - v(tag, "A"); p_t, o_t = sf(c_t)
        c_u = v(tag, "B") - v(tag, "A"); p_u, o_u = sf(c_u)   # unshifted / wrong field
        out["controls"][tag] = {"target": tgt, "contrast_target": o_t, "p_target": p_t,
                                "contrast_KB": o_u, "p_KB": p_u}
        print(f"  {tag:12s} vs K_{tgt:7s}: {o_t:+.3e} p={p_t:.4g}")
        print(f"  {tag:12s} vs K_B     : {o_u:+.3e} p={p_u:.4g}   (should NOT respond)")

    det = [a for a, r in out["amplitudes"].items() if r["p"] < 0.01]
    thr = min(det) if det else None
    out["threshold_alpha"] = thr
    print("\n" + "="*70)
    if thr is not None:
        print(f"CHANNEL THRESHOLD: injected device structure transfers at alpha >= {thr:g}x natural")
        print(f"  => natural PRNU is at least ~{thr:g}x below the demonstrated channel threshold")
    else:
        print(f"NO TRANSFER at any tested amplitude up to {max(C.ALPHAS):g}x natural.")
        print("  => the pipeline does not transmit even a 12x-amplified fixed-coordinate")
        print("     fingerprint; the bound on natural leakage is limited by the CHANNEL,")
        print("     not by measurement sensitivity. Report this explicitly.")
    print("="*70)
    atomic_json_dump(out, os.path.join(EXT_ROOT, "E_AMP_results.json"), indent=2)

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    rr = real.groupby("alpha")["rho_KB"].median()
    ax[0].plot(rr.index, rr.values, "o-", label="real images (reference)")
    ax[0].set_xlabel("injection amplitude α"); ax[0].set_ylabel("median ρ → K_B")
    ax[0].set_title("measurement chain calibration"); ax[0].legend(fontsize=8)
    ax[1].errorbar(xs, ys, fmt="s-", label="generations (paired contrast)")
    ax[1].axhline(0, color="k", lw=0.5)
    ax[1].set_xlabel("injection amplitude α"); ax[1].set_ylabel("ρ(→K_B) − ρ(→K_A)")
    ax[1].set_title("transfer through LoRA + txt2img"); ax[1].legend(fontsize=8)
    fig.savefig(os.path.join(EXT_ROOT, "figs", "eamp_dose_response.png"), dpi=160,
                bbox_inches="tight"); plt.close(fig)
    print("[A5] figure ->", os.path.join(EXT_ROOT, "figs"))

if stage("A5"): a5_analyse()

## Running it

Set `C.STAGES` and re-run the A0 cell each time.

| Session | STAGES | Time | Check |
|---|---|---|---|
| 1 | `("A0","A1")` | ~15 min | NCC(Q, K_B) ≈ 0, NCC(K_B_shift, K_B) ≈ 0, real dose-response **monotone in α** |
| 2 | `("A2",)` | ~3 h (7 adapters) | every ‖lora_B‖ finite and non-zero |
| 3 | `("A3",)` | ~2 h (3500 gens) | all tags complete |
| 4 | `("A4","A5")` | ~15 min | **threshold α** |

**How to read A5.** A genuine transfer must satisfy all of: correct fingerprint (K_B, not K_A),
correct sign, monotone in α, and — critically — the **Q control must not respond**. If Q responds
as strongly as K_B, the model is learning *any* strong fixed-coordinate texture and the result says
nothing about device identity. The shifted control must respond at the shifted alignment and not at
the unshifted one.

**If nothing transfers even at 12×**, that is a stronger result than it first looks: the bound on
natural leakage is then limited by the *channel*, not by your measurement sensitivity, and the
paper says so. Do not present it as a failed experiment.